# Exploring Advanced Prompt Engineering Patterns

In this notebook you will use ChatGPT and LangChain to learn about:

- Chain of Thought Pattern
- Self-Consistency Pattern
- Least to Most Pattern
- ReAct Pattern

___Created By: Dipanjan (DJ)___

## Install OpenAI and LangChain dependencies


In [ ]:
!pip install langchain==0.3.10
!pip install langchain-openai==0.2.12
!pip install langchain-community==0.3.11
!pip install duckduckgo-search==6.3.5
!pip install beautifulsoup4

## Load OpenAI API Credentials

Here we load it from get password function

## Enter API Tokens

In [ ]:
from getpass import getpass

OPENAI_KEY = getpass('Enter Open AI API Key: ')

In [ ]:
import os

os.environ['OPENAI_API_KEY'] = OPENAI_KEY

## Load Necessary Dependencies and ChatGPT LLM

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

## Chain-of-Thought (CoT) Pattern

### Zero Shot CoT

In [ ]:
from IPython.display import display, Markdown

prompt = """Given the following game details,
            How many yards was the shortest valid field goal?
            Think step-by-step to get to your answer.
            Show your step-by-step reasoning and then the answer.

            Coming off their win over the Broncos, the Redskins flew to Cowboys Stadium
            for their Week 11 NFC East rivalry match against the Dallas Cowboys.
            After a scoreless first quarter, Washington would strike in the second quarter
            as kicker Shaun Suisham nailed a 35-yard field goal.
            The Redskins would try to add onto their lead in the third quarter
            with Suisham booting a 31-yard field goal which was blocked.
            However, in the fourth quarter, the Cowboys rallied as quarterback Tony Romo
            completing a 10-yard touchdown pass to wide receiver Patrick Crayton.
         """

response = chatgpt.invoke(prompt)
display(Markdown(response.content))

### Few-Shot CoT

In [ ]:
from IPython.display import display, Markdown

prompt = """Given the following game details,
            How many yards was the shortest valid field goal?
            Think step-by-step to get to your answer similar to the following examples:

            Example 1:
            Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls.
               Each can has 3 tennis balls. How many tennis balls does he have now?

            Reasoning:
              Step 1: Roger started with 5 balls.
              Step 2: 2 cans of 3 tennis balls each is 6 tennis balls in total.
              Step 3: 5 + 6 = 11 tennis balls.

            A: 11 Tennis Balls

            Example 2:
            Q: How many keystrokes are needed to type the numbers from 1 to 500?

            Reasoning:
              Step 1: There are 9 one-digit numbers from 1 to 9.
              Step 2: There are 90 two-digit numbers from 10 to 99.
              Step 3: There are 401 three-digit numbers from 100 to 500.
              Step 4: Adding all of them up, 9 + 90 + 401 = 1392 keystrokes.

            A: 1392 keystrokes


            Q: Coming off their win over the Broncos, the Redskins flew to Cowboys Stadium
            for their Week 11 NFC East rivalry match against the Dallas Cowboys.
            After a scoreless first quarter, Washington would strike in the second quarter
            as kicker Shaun Suisham nailed a 35-yard field goal.
            The Redskins would try to add onto their lead in the third quarter
            with Suisham booting a 31-yard field goal which was blocked.
            However, in the fourth quarter, the Cowboys rallied as quarterback Tony Romo
            completing a 10-yard touchdown pass to wide receiver Patrick Crayton.
         """

response = chatgpt.invoke(prompt)
print(response.content)

## Self-Consistency Pattern

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

cot_prompt_txt = """Given the following problem,
                    Think step-by-step to get to your answer similar to the following example:

                    Example 1:
                    Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls.
                    Each can has 3 tennis balls. How many tennis balls does he have now?

                    Reasoning:
                    Step 1: Roger started with 5 balls.
                    Step 2: 2 cans of 3 tennis balls each is 6 tennis balls in total.
                    Step 3: 5 + 6 = 11 tennis balls.

                    A: 11 Tennis Balls

                    Problem:
                    {problem}
                """

cot_prompt = ChatPromptTemplate.from_template(cot_prompt_txt)
cot_prompt.pretty_print()

In [ ]:
self_con_prompt_txt = """Given the following problem
                         and 3 diverse reasoning paths explored by an AI model
                         analyse these pathways carefully,
                         aggregate, take the majority vote as needed
                         and generate a final single reasoning path along with the answer

                         AI Model Reasoning Path 1:
                         {reasoning_path_1}

                         AI Model Reasoning Path 2:
                         {reasoning_path_2}

                         AI Model Reasoning Path 3:
                         {reasoning_path_3}

                         Problem:
                         {problem}
                    """

self_con_prompt = ChatPromptTemplate.from_template(self_con_prompt_txt)
self_con_prompt.pretty_print()

In [ ]:
# do CoT exploration independently 3 times with these models
gpt1 = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)
gpt2 = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.5)
gpt3 = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.9)

In [ ]:
# do final reasoning and aggregation with this model
chatgpt = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# independent CoT explorations with 3 different LLMs (with different temp settings)
llm_chain1 = cot_prompt | gpt1 | StrOutputParser()
llm_chain2 = cot_prompt | gpt2 | StrOutputParser()
llm_chain3 = cot_prompt | gpt3 | StrOutputParser()

sc_chain = (
                RunnableParallel(problem=RunnablePassthrough(),
                                 reasoning_path_1=llm_chain1,
                                 reasoning_path_2=llm_chain2,
                                 reasoning_path_3=llm_chain3)
                        |
                self_con_prompt
                        |
                    chatgpt
            )

In [ ]:
question = """Q: Coming off their win over the Broncos, the Redskins flew to Cowboys Stadium
            for their Week 11 NFC East rivalry match against the Dallas Cowboys.
            After a scoreless first quarter, Washington would strike in the second quarter
            as kicker Shaun Suisham nailed a 35-yard field goal.
            The Redskins would try to add onto their lead in the third quarter
            with Suisham booting a 31-yard field goal which was blocked.
            However, in the fourth quarter, the Cowboys rallied as quarterback Tony Romo
            completing a 10-yard touchdown pass to wide receiver Patrick Crayton.
            How many yards was the shortest valid field goal?
           """

In [ ]:
response = sc_chain.invoke({'problem': question})
display(Markdown(response.content))

## Least to Most Prompting

In [ ]:
PROMPT = """You are a strong reasoning agent.
            Respond to user question with concise and helpful information.
            Follow the format as mentioned in the example,
            break down the problem first into sub problems,
            answer each sub problem sequentially, analyze it
            and then get to the final answer.
            Do not jump directly to the answer

            Example:

            Q: Against Tim Tebow and the Broncos, the two teams would be in a 0-0 deadlock
            in the first half, though the Broncos would nearly score in the second quarter
            on a 28-yard field goal, which would then be blocked by Julius Peppers.
            The Bears would then score 10 points on Marion Barber's 9-yard touchdown run,
            and Robbie Gould's team record-breaking 57-yard field goal, but Tebow's touchdown pass
            to Demaryius Thomas and Matt Prater's 59-yard field goal would tie the game.
            Barber would commit two costly mistakes during the late portion of the game.
            In the fourth quarter, Barber would run out of bounds with 1:55 left,
            and Barber would also fumble in overtime.
            The Broncos would then move downfield and kick a game-winning field goal.
            How many yards was the games longest field goal?

            Response:
            Let's break down this problem into subproblems:
              1. What were the field goals when field goal was?
              2. How many yards was the games longest field goal?

            The answer to subproblems are as follows:
              1. The field goals were 28-yard,  57-yard and  59-yard which are all valid goals
              2. The maximum value out of  59-yard, 28-yard and 57-yard is 59

            A: The final answer is 59 yards.

            Question:
            {query}
        """

query = """Coming off their win over the Broncos, the Redskins flew to Cowboys Stadium
           for their Week 11 NFC East rivalry match against the Dallas Cowboys.
           After a scoreless first quarter, Washington would strike in the second quarter
           as kicker Shaun Suisham nailed a 35-yard field goal.
           The Redskins would try to add onto their lead in the third quarter
           with Suisham booting a 31-yard field goal which was blocked.
           However, in the fourth quarter, the Cowboys rallied as quarterback Tony Romo
           completing a 10-yard touchdown pass to wide receiver Patrick Crayton.
           How many yards was the shortest valid field goal?
        """

prompt = ChatPromptTemplate.from_template(PROMPT)

chain = (prompt
           |
         chatgpt
)

response = chain.invoke({"query": query})
print(response.content)

## ReAct Prompting - Agentic AI Flow

In [ ]:
prompt = """Tell me about LangGraph in detail including who made it and key features.
            Do not make up answers, only answer if you are sure"""

# it will either not answer or end up hallucinating the answer
response = chatgpt.invoke(prompt)
display(Markdown(response.content))

In [ ]:
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import tool
from bs4 import BeautifulSoup
import requests
from tqdm import tqdm

@tool
def search_web(query: str) -> list:
    """Search the web for a query."""
    print('Searching web and extracting page info')
    wrapper = DuckDuckGoSearchAPIWrapper(max_results=5)
    results = wrapper.results(query, max_results=5)
    docs = []
    for result in tqdm(results):
        # Sending a request to the URL
        response = requests.get(result['link'])
        # Parsing the page content
        soup = BeautifulSoup(response.content, 'html.parser')
        # Extracting all text content from the page
        text_content = soup.get_text(separator="\n", strip=True)
        docs.append(text_content)
    return docs

In [ ]:
r = search_web('What is langgraph')

In [ ]:
print(r[0][:1000])

In [ ]:
chatgpt = ChatOpenAI(model="gpt-4o", temperature=0)
tools = [search_web]
chatgpt_with_tools = chatgpt.bind_tools(tools)
prompt = "What is langgraph?"
response = chatgpt_with_tools.invoke(prompt)
response

In [ ]:
response.tool_calls

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYS_PROMPT = """You run in a loop of Thought, Action, PAUSE, Observation.
                At the end of the loop, you output an Answer.
                Use Thought to describe your thoughts about the question you have been asked.
                Use Action to run one of the actions available to you - then return PAUSE.
                Observation will be the result of running those actions.
                For the user query below, first break it into multiple sub queries or sub steps if needed.
                The follow the below workflow for each sub query or sub step
                Analyze, validate the relevant results and then compile to generate the final answer

                Use the following workflow format:
                Question: the input task you must solve
                Thought: you should always think about what to do
                Action: the action to take:
                          - should be either to break down the task into sub tasks first if needed
                          - refer to your trained knowledge or call the tool [search_web]
                            if you need to get additional information from the web
                Action Input: the input to the action
                Observation: the result of the action
                ... (this Thought/Action/Action Input/Observation can repeat N times)
                Thought: I now know the final answer
                Final Answer: the final answer to the original input question
             """

prompt_template = ChatPromptTemplate.from_messages(
   [
       ("system", SYS_PROMPT),
       ("human", "Query: {query}"),
       MessagesPlaceholder(variable_name="agent_scratchpad"),
   ]
)

In [ ]:
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

tools = [search_web]
agent = create_tool_calling_agent(chatgpt, tools, prompt_template)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
prompt = """Tell me about LangGraph in detail including who made it and key features.
            Do not guess answers, only answer if you are sure"""
response = agent_executor.invoke({"query": prompt})

In [ ]:
from IPython.display import display, Markdown

display(Markdown(response['output']))